In [61]:
from sympy.physics.paulialgebra import Pauli
from sympy.physics.quantum import TensorProduct, Commutator
from sympy import I, simplify, symbols, diff, Function


from itertools import combinations, permutations

from sequence_2_matrix import A_direct_mat, b_direct_vec, solve_cd_torch
from utils_test import tp_trace, tp_compose

import torch
import numpy as np

In [62]:
n_qubits = 3  # change to 3, 4, ...

# Pauli matrices as symbols (algebraic)
s1 = Pauli(1)  # σˣ
s2 = Pauli(2)  # σʸ
s3 = Pauli(3)  # σᶻ

I2 = Pauli(1) * Pauli(1)
def single_body_op(pauli, site, n):
    factors = [I2] * n
    factors[site] = pauli
    return TensorProduct(*factors)


def two_body_op(pauli1, pauli2, site1, site2, n):
    factors = [I2] * n
    factors[site1] = pauli1
    factors[site2] = pauli2
    return TensorProduct(*factors)


nop = (I2 - s1) / 2  # projector onto |r⟩, we will this later
ni = [single_body_op(nop, i, n_qubits) for i in range(n_qubits)]

# Single-site Pauli operators: X[i], Y[i], Z[i]  (0-indexed)
X = [single_body_op(s1, i, n_qubits) for i in range(n_qubits)]
Y = [single_body_op(s2, i, n_qubits) for i in range(n_qubits)]
Z = [single_body_op(s3, i, n_qubits) for i in range(n_qubits)]

# Per-qubit drive amplitudes, phases, detunings
Omega = [symbols(f"Omega_{i+1}", real=True, positive=True) for i in range(n_qubits)]
Mu = [symbols(f"mu_{i+1}", real=True) for i in range(n_qubits)]
Nu = [symbols(f"nu_{i+1}", real=True) for i in range(n_qubits)]

# van der Waals couplings for each pair i < j
U = {
    (i, j): symbols(f"U{i+1}{j+1}", real=True, positive=True)
    for i, j in combinations(range(n_qubits), 2)
}

# Adiabatic scheduling parameter
lam = symbols("lambda", real=True)

# ── CD single-body coefficients ───────────────────────────────────────────
a = {i: symbols(f"a_{i+1}", real=True) for i in range(n_qubits)}
b = {i: symbols(f"b_{i+1}", real=True) for i in range(n_qubits)}
c = {i: symbols(f"c_{i+1}", real=True) for i in range(n_qubits)}

# ── CD two-body symmetric (same Pauli): unordered pairs i < j ────────────
d_xx = {
    (i, j): symbols(f"delta_xx_{i+1}{j+1}", real=True)
    for i, j in combinations(range(n_qubits), 2)
}
d_yy = {
    (i, j): symbols(f"delta_yy_{i+1}{j+1}", real=True)
    for i, j in combinations(range(n_qubits), 2)
}
d_zz = {
    (i, j): symbols(f"delta_zz_{i+1}{j+1}", real=True)
    for i, j in combinations(range(n_qubits), 2)
}

# ── CD two-body asymmetric (mixed Paulis): all ordered pairs i ≠ j ────────
# d_xy[(i,j)] = coeff of X_i Y_j
d_xy = {
    (i, j): symbols(f"delta_xy_{i+1}{j+1}", real=True)
    for i, j in permutations(range(n_qubits), 2)
}
d_xz = {
    (i, j): symbols(f"delta_xz_{i+1}{j+1}", real=True)
    for i, j in permutations(range(n_qubits), 2)
}
d_yz = {
    (i, j): symbols(f"delta_yz_{i+1}{j+1}", real=True)
    for i, j in permutations(range(n_qubits), 2)
}

# ── λ-dependent schedule functions ───────────────────────────────────────
Omega_l = [Function(f"Omega_{i+1}")(lam) for i in range(n_qubits)]
mu_l = [Function(f"mu_{i+1}")(lam) for i in range(n_qubits)]
nu_l = [Function(f"nu_{i+1}")(lam) for i in range(n_qubits)]

A = sum(a[i] * X[i] + b[i] * Y[i] + c[i] * Z[i] for i in range(n_qubits))
for i, j in combinations(range(n_qubits), 2):
    A += d_xx[(i, j)] * two_body_op(s1, s1, i, j, n_qubits)
    A += d_yy[(i, j)] * two_body_op(s2, s2, i, j, n_qubits)
    A += d_zz[(i, j)] * two_body_op(s3, s3, i, j, n_qubits)
for i, j in permutations(range(n_qubits), 2):
    A += d_xy[(i, j)] * two_body_op(s1, s2, i, j, n_qubits)
    A += d_xz[(i, j)] * two_body_op(s1, s3, i, j, n_qubits)
    A += d_yz[(i, j)] * two_body_op(s2, s3, i, j, n_qubits)

In [63]:
A

a_1*sigma1x1x1 + a_2*1xsigma1x1 + a_3*1x1xsigma1 + b_1*sigma2x1x1 + b_2*1xsigma2x1 + b_3*1x1xsigma2 + c_1*sigma3x1x1 + c_2*1xsigma3x1 + c_3*1x1xsigma3 + delta_xx_12*sigma1xsigma1x1 + delta_xx_13*sigma1x1xsigma1 + delta_xx_23*1xsigma1xsigma1 + delta_xy_12*sigma1xsigma2x1 + delta_xy_13*sigma1x1xsigma2 + delta_xy_21*sigma2xsigma1x1 + delta_xy_23*1xsigma1xsigma2 + delta_xy_31*sigma2x1xsigma1 + delta_xy_32*1xsigma2xsigma1 + delta_xz_12*sigma1xsigma3x1 + delta_xz_13*sigma1x1xsigma3 + delta_xz_21*sigma3xsigma1x1 + delta_xz_23*1xsigma1xsigma3 + delta_xz_31*sigma3x1xsigma1 + delta_xz_32*1xsigma3xsigma1 + delta_yy_12*sigma2xsigma2x1 + delta_yy_13*sigma2x1xsigma2 + delta_yy_23*1xsigma2xsigma2 + delta_yz_12*sigma2xsigma3x1 + delta_yz_13*sigma2x1xsigma3 + delta_yz_21*sigma3xsigma2x1 + delta_yz_23*1xsigma2xsigma3 + delta_yz_31*sigma3x1xsigma2 + delta_yz_32*1xsigma3xsigma2 + delta_zz_12*sigma3xsigma3x1 + delta_zz_13*sigma3x1xsigma3 + delta_zz_23*1xsigma3xsigma3

In [64]:
# ── Define the Hamiltonian ───────────────────────────────────────────────
# ── Static Hamiltonian ────────────────────────────────────────────────────
H = sum(
    Omega[i] * X[i] + Mu[i] * Y[i] + Nu[i] * Z[i]
    for i in range(n_qubits)
) + sum(
    U[(i,j)] * two_body_op(s3, s3, i, j, n_qubits)
    for i,j in combinations(range(n_qubits), 2)
)

# ── λ-dependent Hamiltonian (for dH/dλ) ──────────────────────────────────
H_lam = sum(
    Omega_l[i] * X[i] + mu_l[i] * Y[i] + nu_l[i] * Z[i]
    for i in range(n_qubits)
) + sum(
    U[(i,j)] * two_body_op(s3, s3, i, j, n_qubits)
    for i,j in combinations(range(n_qubits), 2)
)

dH_dlam = diff(H_lam, lam)


In [65]:
def make_cd_coeffs(n):
    """Generate all 2-body CD coefficients for n qubits."""
    a = {i: symbols(f'a_{i+1}', real=True) for i in range(n)}
    b = {i: symbols(f'b_{i+1}', real=True) for i in range(n)}
    c = {i: symbols(f'c_{i+1}', real=True) for i in range(n)}

    d_xx = {(i,j): symbols(f'delta_xx_{i+1}{j+1}', real=True) for i,j in combinations(range(n), 2)}
    d_yy = {(i,j): symbols(f'delta_yy_{i+1}{j+1}', real=True) for i,j in combinations(range(n), 2)}
    d_zz = {(i,j): symbols(f'delta_zz_{i+1}{j+1}', real=True) for i,j in combinations(range(n), 2)}

    d_xy = {(i,j): symbols(f'delta_xy_{i+1}{j+1}', real=True) for i,j in permutations(range(n), 2)}
    d_xz = {(i,j): symbols(f'delta_xz_{i+1}{j+1}', real=True) for i,j in permutations(range(n), 2)}
    d_yz = {(i,j): symbols(f'delta_yz_{i+1}{j+1}', real=True) for i,j in permutations(range(n), 2)}

    return a, b, c, d_xx, d_yy, d_zz, d_xy, d_xz, d_yz


def make_ansatz(n):
    """Build ansatz A as a sum of scalar * TensorProduct terms."""
    a, b, c, d_xx, d_yy, d_zz, d_xy, d_xz, d_yz = make_cd_coeffs(n)

    A = sum(
          a[i] * single_body_op(s1, i, n)
        + b[i] * single_body_op(s2, i, n)
        + c[i] * single_body_op(s3, i, n)
        for i in range(n)
    )
    for i,j in combinations(range(n), 2):
        A += d_xx[(i,j)] * two_body_op(s1, s1, i, j, n)
        A += d_yy[(i,j)] * two_body_op(s2, s2, i, j, n)
        A += d_zz[(i,j)] * two_body_op(s3, s3, i, j, n)
    for i,j in permutations(range(n), 2):
        A += d_xy[(i,j)] * two_body_op(s1, s2, i, j, n)
        A += d_xz[(i,j)] * two_body_op(s1, s3, i, j, n)
        A += d_yz[(i,j)] * two_body_op(s2, s3, i, j, n)

    return A, a, b, c, d_xx, d_yy, d_zz, d_xy, d_xz, d_yz


# ── Tests ─────────────────────────────────────────────────────────────────

for n in [2, 3, 4]:
    a, b, c, d_xx, d_yy, d_zz, d_xy, d_xz, d_yz = make_cd_coeffs(n)

    n_single = 3 * n
    n_sym    = 3 * len(list(combinations(range(n), 2)))   # 3 * C(n,2)
    n_asym   = 3 * len(list(permutations(range(n), 2)))   # 3 * n*(n-1)
    n_total  = n_single + n_sym + n_asym

    print(f"n={n}: single={n_single}, sym={n_sym}, asym={n_asym}, total={n_total}  (4^n-1={4**n-1}) with 3 body terms")

    # spot-check: correct symbol names
    assert a[0].name == 'a_1'
    assert d_xx[(0,1)].name == 'delta_xx_12'
    assert d_xy[(1,0)].name == 'delta_xy_21'
    if n >= 3:
        assert d_yz[(2,0)].name == 'delta_yz_31'

    # no duplicate symbols
    all_syms = (list(a.values()) + list(b.values()) + list(c.values()) +
                list(d_xx.values()) + list(d_yy.values()) + list(d_zz.values()) +
                list(d_xy.values()) + list(d_xz.values()) + list(d_yz.values()))
    assert len(all_syms) == len(set(s.name for s in all_syms)), "duplicate symbols!"

    print(f"     all assertions passed")
    print(f"     sample symbols: {a[0]}, {d_xx[(0,1)]}, {d_xy[(1,0)]}")


n=2: single=6, sym=3, asym=6, total=15  (4^n-1=15) with 3 body terms
     all assertions passed
     sample symbols: a_1, delta_xx_12, delta_xy_21
n=3: single=9, sym=9, asym=18, total=36  (4^n-1=63) with 3 body terms
     all assertions passed
     sample symbols: a_1, delta_xx_12, delta_xy_21
n=4: single=12, sym=18, asym=36, total=66  (4^n-1=255) with 3 body terms
     all assertions passed
     sample symbols: a_1, delta_xx_12, delta_xy_21


In [66]:
icommutator = (I*Commutator(A,H).expand(commutator=True)).expand(commutator=True).doit()

In [67]:
CD = dH_dlam + icommutator
icommutator

2*Omega_1*b_1*sigma3x1x1 - 2*Omega_1*c_1*sigma2x1x1 + 2*Omega_1*delta_xy_21*sigma3xsigma1x1 + 2*Omega_1*delta_xy_31*sigma3x1xsigma1 - 2*Omega_1*delta_xz_21*sigma2xsigma1x1 - 2*Omega_1*delta_xz_31*sigma2x1xsigma1 + 2*Omega_1*delta_yy_12*sigma3xsigma2x1 + 2*Omega_1*delta_yy_13*sigma3x1xsigma2 + 2*Omega_1*delta_yz_12*sigma3xsigma3x1 + 2*Omega_1*delta_yz_13*sigma3x1xsigma3 - 2*Omega_1*delta_yz_21*sigma2xsigma2x1 - 2*Omega_1*delta_yz_31*sigma2x1xsigma2 - 2*Omega_1*delta_zz_12*sigma2xsigma3x1 - 2*Omega_1*delta_zz_13*sigma2x1xsigma3 + 2*Omega_2*b_2*1xsigma3x1 - 2*Omega_2*c_2*1xsigma2x1 + 2*Omega_2*delta_xy_12*sigma1xsigma3x1 + 2*Omega_2*delta_xy_32*1xsigma3xsigma1 - 2*Omega_2*delta_xz_12*sigma1xsigma2x1 - 2*Omega_2*delta_xz_32*1xsigma2xsigma1 + 2*Omega_2*delta_yy_12*sigma2xsigma3x1 + 2*Omega_2*delta_yy_23*1xsigma3xsigma2 - 2*Omega_2*delta_yz_12*sigma2xsigma2x1 + 2*Omega_2*delta_yz_21*sigma3xsigma3x1 + 2*Omega_2*delta_yz_23*1xsigma3xsigma3 - 2*Omega_2*delta_yz_32*1xsigma2xsigma2 - 2*Omega_2*de

In [68]:
if n_qubits == 2:
    expected = -Omega[0]*d_yz[(1,0)]-Omega[1]*d_yz[(0,1)]+d_xy[(0,1)]*Nu[0]+d_xy[(1,0)]*Nu[1]
    assert simplify(tp_trace(tp_compose(Y[0]*Y[1], CD),n_qubits)/2**(n_qubits+1) - expected) == 0
    
    expected = -b[0]*Nu[0]+c[0]*Mu[0]-d_yz[(0,1)]*U[(0,1)]+diff(Omega_l[0],lam)/2
    assert (tp_trace(tp_compose(X[0], CD),n_qubits)/2**(n_qubits+1)).equals(expected)
    
    expected = a[1]*Nu[1]-c[1]*Omega[1]+d_xz[(1,0)]*U[(0,1)]+diff(mu_l[1],lam)/2
    assert (tp_trace(tp_compose(Y[1], CD),n_qubits)/2**(n_qubits+1)).equals(expected)


In [69]:
tp_trace(tp_compose(X[0]*Y[1]*Y[2], CD),n_qubits)/2**(n_qubits+1)

0

In [70]:
if n_qubits == 3:
    expected = -Omega[0]*d_yz[(1,0)]-Omega[1]*d_yz[(0,1)]+d_xy[(0,1)]*Nu[0]+d_xy[(1,0)]*Nu[1]
    assert simplify(tp_trace(tp_compose(Y[0]*Y[1], CD),n_qubits)/2**(n_qubits+1) - expected) == 0

    expected = -Omega[1]*d_yz[(2,1)]-Omega[2]*d_yz[(1,2)]+d_xy[(1,2)]*Nu[1]+d_xy[(2,1)]*Nu[2]
    assert simplify(tp_trace(tp_compose(Y[1]*Y[2], CD),n_qubits)/2**(n_qubits+1)- expected) == 0

    # 3 qubit terms
    expected = -U[(0,2)]*d_yy[(0,1)]+U[(1,2)]*d_xx[(0,1)]
    assert simplify(tp_trace(tp_compose(X[0]*Y[1]*Z[2], CD),n_qubits)/2**(n_qubits+1) - expected) == 0
    #3 qubit terms
    expected = -U[(0,1)]*d_yz[(0,2)]-U[(0,2)]*d_yz[(0,1)]
    assert simplify(tp_trace(tp_compose(X[0]*Z[1]*Z[2], CD),n_qubits)/2**(n_qubits+1) - expected) == 0



In [71]:
tp_trace(tp_compose(X[2], CD),n_qubits)/2**(n_qubits+1) 

-U13*delta_yz_31 - U23*delta_yz_32 - b_3*nu_3 + c_3*mu_3 + Derivative(Omega_3(lambda), lambda)/2

In [72]:
# ── Pauli basis (same ordering as ansatz A) ──────────────────────────────
def make_basis_ops(n):
    """Return (ops, eq_labels, x_labels): rows=equations, cols=unknowns."""
    ops, eq_labels, x_labels = [], [], []
    def paired_permutations(n):
        return [(i, j) for i, j in combinations(range(n), 2) for i, j in [(i,j),(j,i)]]
    
    for i in range(n):
        ops.append(single_body_op(s1, i, n)); eq_labels.append(f'X{i+1}'); x_labels.append(f'a_{i+1}')
        ops.append(single_body_op(s2, i, n)); eq_labels.append(f'Y{i+1}'); x_labels.append(f'b_{i+1}')
        ops.append(single_body_op(s3, i, n)); eq_labels.append(f'Z{i+1}'); x_labels.append(f'c_{i+1}')
    for i, j in combinations(range(n), 2):
        ops.append(two_body_op(s1, s1, i, j, n)); eq_labels.append(f'X{i+1}X{j+1}'); x_labels.append(f'dxx_{i+1}{j+1}')
        ops.append(two_body_op(s2, s2, i, j, n)); eq_labels.append(f'Y{i+1}Y{j+1}'); x_labels.append(f'dyy_{i+1}{j+1}')
        ops.append(two_body_op(s3, s3, i, j, n)); eq_labels.append(f'Z{i+1}Z{j+1}'); x_labels.append(f'dzz_{i+1}{j+1}')
    for i, j in paired_permutations(n):
        ops.append(two_body_op(s1, s2, i, j, n)); eq_labels.append(f'X{i+1}Y{j+1}'); x_labels.append(f'dxy_{i+1}{j+1}')
        ops.append(two_body_op(s1, s3, i, j, n)); eq_labels.append(f'X{i+1}Z{j+1}'); x_labels.append(f'dxz_{i+1}{j+1}')
        ops.append(two_body_op(s2, s3, i, j, n)); eq_labels.append(f'Y{i+1}Z{j+1}'); x_labels.append(f'dyz_{i+1}{j+1}')
    return ops, eq_labels, x_labels


# ── Build constant coefficient tensors (symbolic, done once) ─────────────


def build_coefficient_tensors(n):
    """
    Exploits linearity of H to decompose M and b into parameter-free matrices:

        M(Ω,μ,ν,U) = Σ_k [ Ω_k·M_Om[k] + μ_k·M_Mu[k] + ν_k·M_Nu[k] ]
                     + Σ_{k<l} U_kl · M_U[(k,l)]

        b(Ω̇,μ̇,ν̇)  = Σ_k [ Ω̇_k·b_Om[k] + μ̇_k·b_Mu[k] + ν̇_k·b_Nu[k] ]

    where M[i,j] = Tr(P_i · i[P_j, H_unit]) / 2^(n+1)  (one H term at a time)
          b[i]   = -Tr(P_i · dH_unit)       / 2^(n+1)

    Each returned object is a constant float64 torch tensor (no grad).
    """
    basis_ops, _, _ = make_basis_ops(n)
    norm = 2 ** (n + 1)
    N = len(basis_ops)

    def _M_for(H_unit):
        mat = np.zeros((N, N))
        for i, P_i in enumerate(basis_ops):
            for j, P_j in enumerate(basis_ops):
                comm = (I * Commutator(P_j, H_unit)
                        .expand(commutator=True)).expand(commutator=True).doit()
                if comm == 0:
                    continue
                mat[i, j] = float(tp_trace(tp_compose(P_i, comm), n) / norm)
        return torch.tensor(mat, dtype=torch.float64)

    def _b_for(dH_unit):
        vec = np.zeros(N)
        for i, P_i in enumerate(basis_ops):
            entry = -tp_trace(tp_compose(P_i, dH_unit), n) / norm
            if entry == 0:
                continue
            vec[i] = float(entry)
        return torch.tensor(vec, dtype=torch.float64)

    print(f"Building coefficient tensors  n={n}  N={N} ...")
    M_Om = [_M_for(single_body_op(s1, k, n)) for k in range(n)]
    M_Mu = [_M_for(single_body_op(s2, k, n)) for k in range(n)]
    M_Nu = [_M_for(single_body_op(s3, k, n)) for k in range(n)]
    M_U  = {(i, j): _M_for(two_body_op(s3, s3, i, j, n)) for i, j in combinations(range(n), 2)}
    b_Om = [_b_for(single_body_op(s1, k, n)) for k in range(n)]
    b_Mu = [_b_for(single_body_op(s2, k, n)) for k in range(n)]
    b_Nu = [_b_for(single_body_op(s3, k, n)) for k in range(n)]
    return M_Om, M_Mu, M_Nu, M_U, b_Om, b_Mu, b_Nu


def make_M_torch(M_Om, M_Mu, M_Nu, M_U, Omega, Mu, Nu, U):
    """Assemble M — gradients flow through Omega/Mu/Nu/U."""
    M = sum(Omega[k] * M_Om[k] + Mu[k] * M_Mu[k] + Nu[k] * M_Nu[k] for k in range(len(Omega)))
    for (i, j), M_ij in M_U.items():
        M = M + U[(i, j)] * M_ij
    return M


def make_b_torch(b_Om, b_Mu, b_Nu, dOmega, dMu, dNu):
    """Assemble b — gradients flow through dOmega/dMu/dNu."""
    return sum(dOmega[k] * b_Om[k] + dMu[k] * b_Mu[k] + dNu[k] * b_Nu[k] for k in range(len(dOmega)))



In [73]:
print(make_basis_ops(3)[0][18:])
print(make_basis_ops(3)[1][18:])
print(make_basis_ops(3)[2][18:])
    

[sigma1xsigma2x1, sigma1xsigma3x1, sigma2xsigma3x1, sigma2xsigma1x1, sigma3xsigma1x1, sigma3xsigma2x1, sigma1x1xsigma2, sigma1x1xsigma3, sigma2x1xsigma3, sigma2x1xsigma1, sigma3x1xsigma1, sigma3x1xsigma2, 1xsigma1xsigma2, 1xsigma1xsigma3, 1xsigma2xsigma3, 1xsigma2xsigma1, 1xsigma3xsigma1, 1xsigma3xsigma2]
['X1Y2', 'X1Z2', 'Y1Z2', 'X2Y1', 'X2Z1', 'Y2Z1', 'X1Y3', 'X1Z3', 'Y1Z3', 'X3Y1', 'X3Z1', 'Y3Z1', 'X2Y3', 'X2Z3', 'Y2Z3', 'X3Y2', 'X3Z2', 'Y3Z2']
['dxy_12', 'dxz_12', 'dyz_12', 'dxy_21', 'dxz_21', 'dyz_21', 'dxy_13', 'dxz_13', 'dyz_13', 'dxy_31', 'dxz_31', 'dyz_31', 'dxy_23', 'dxz_23', 'dyz_23', 'dxy_32', 'dxz_32', 'dyz_32']


In [74]:
# ── Step 1: build constant tensors once (slow, symbolic) ─────────────────

n_cd = 3  # change to 3 for full system

M_Om, M_Mu, M_Nu, M_U, b_Om, b_Mu, b_Nu = build_coefficient_tensors(n_cd)
_, _, x_labels_cd = make_basis_ops(n_cd)

# ── Step 2: torch tensors with requires_grad=True ─────────────────────────
# Hamiltonian parameters (fixed over lambda sweep, grad flows here)
Omega_t = [
    torch.tensor(3.0+0.1*k, dtype=torch.float64, requires_grad=True) for k in range(n_cd)
]
Mu_t = [torch.tensor(2.0+0.1*k, dtype=torch.float64, requires_grad=True) for k in range(n_cd)]
Nu_t = [torch.tensor(1.0+0.1*k, dtype=torch.float64, requires_grad=True) for k in range(n_cd)]
U_t = {
    (i, j): torch.tensor(7.0+0.1*i+0.01*j, dtype=torch.float64, requires_grad=True)
    for i, j in combinations(range(n_cd), 2)
}


# Schedule derivatives at current lambda (change per lambda step)

dOmega_t = [
    torch.tensor(0.3 * (-1) ** k, dtype=torch.float64, requires_grad=True)
    for k in range(n_cd)
]
dMu_t = [
    torch.tensor(0.2, dtype=torch.float64, requires_grad=True) for _ in range(n_cd)
]
dNu_t = [
    torch.tensor(0.1 * (-1) ** k, dtype=torch.float64, requires_grad=True)
    for k in range(n_cd)
]


# ── Step 3: assemble and solve — full autograd graph ─────────────────────
M_t = make_M_torch(M_Om, M_Mu, M_Nu, M_U, Omega_t, Mu_t, Nu_t, U_t)
b_t = make_b_torch(b_Om, b_Mu, b_Nu, dOmega_t, dMu_t, dNu_t)


Building coefficient tensors  n=3  N=36 ...


In [75]:
print(dOmega_t)
b_t

[tensor(0.3000, dtype=torch.float64, requires_grad=True), tensor(-0.3000, dtype=torch.float64, requires_grad=True), tensor(0.3000, dtype=torch.float64, requires_grad=True)]


tensor([-0.1500, -0.1000, -0.0500,  0.1500, -0.1000,  0.0500, -0.1500, -0.1000,
        -0.0500,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000], dtype=torch.float64,
       grad_fn=<AddBackward0>)

In [76]:
## M_t is correct and create A_matrix from the sequece

n_atoms = 3
n_single = 3 * n_atoms 
n_sym    = 3 * len(list(combinations(range(n_atoms), 2)))   # 3 * C(n,2)
n_asym   = 3 * len(list(permutations(range(n_atoms), 2)))   # 3 * n*(n-1)
n_total  = n_single + n_sym + n_asym

Omega_t = [
    torch.tensor(3.0+0.1*k, dtype=torch.float64,requires_grad=True) for k in range(n_atoms)
]
Mu_t = [torch.tensor(2.0+0.1*k, dtype=torch.float64,requires_grad=True) for k in range(n_atoms)]
Nu_t = [torch.tensor(1.0+0.1*k, dtype=torch.float64,requires_grad=True) for k in range(n_atoms)]

U_t = torch.zeros(n_atoms,n_atoms,dtype=torch.float64)
for i,j in combinations(range(n_atoms), 2):
    U_t[i,j] = torch.tensor(7.0+0.1*i+0.01*j, dtype=torch.float64,)

# Schedule derivatives at current lambda (change per lambda step)

dOmega_t = [
    torch.tensor(0.3 * (-1) ** k, dtype=torch.float64, requires_grad=True)
    for k in range(n_atoms)
]
dMu_t = [
    torch.tensor(0.2, dtype=torch.float64, requires_grad=True) for _ in range(n_atoms)
]
dNu_t = [
    torch.tensor(0.1 * (-1) ** k, dtype=torch.float64, requires_grad=True)
    for k in range(n_atoms)
]

A_matrix = A_direct_mat(n_atoms, Omega_t, Mu_t, Nu_t, U_t)
assert torch.allclose(A_matrix,M_t)


b_vector = b_direct_vec(n_atoms, dOmega_t, dMu_t, dNu_t)
    
assert torch.allclose(b_vector,b_t)
U_t


tensor([[0.0000, 7.0100, 7.0200],
        [0.0000, 0.0000, 7.1200],
        [0.0000, 0.0000, 0.0000]], dtype=torch.float64)

In [77]:
# # soves the system A_matrix x = b_vector, using paulis

x_t = solve_cd_torch(M_t, b_t)

# print("CD coefficients x:")
# for lbl, val in zip(x_labels_cd, x_t.tolist()):
#     print(f"  {lbl:20s} = {val:+.6f}")

# # ── Step 4: backprop through the CD solution ──────────────────────────────
# # Example: gradient of ||x||^2 w.r.t. all parameters
# loss = (x_t**2).sum()
# loss.backward()

In [78]:

#print(f"\nloss = ||x||^2 = {loss.item():.6f}")
# print(f"d(loss)/d(Omega_1)      = {Omega_t[0].grad.item():+.6f}")
# print(f"d(loss)/d(Omega_2)      = {Omega_t[1].grad.item():+.6f}")
# print(f"d(loss)/d(dOmega_1/dlam)= {dOmega_t[0].grad.item():+.6f}")


In [79]:
# using direct A and b formation

x_t_Amat = solve_cd_torch(A_matrix, b_vector)
print("CD coefficients x:")
for lbl, val in zip(x_labels_cd, x_t_Amat.tolist()):
    print(f"  {lbl:20s} = {val:+.6f}")

assert torch.allclose(x_t, x_t_Amat)

CD coefficients x:
  a_1                  = +0.001712
  b_1                  = -0.005882
  c_1                  = -0.006054
  a_2                  = -0.010171
  b_2                  = +0.012012
  c_2                  = +0.028773
  a_3                  = +0.001746
  b_3                  = -0.005325
  c_3                  = -0.007669
  dxx_12               = -0.014672
  dyy_12               = +0.010141
  dzz_12               = +0.000127
  dxx_13               = +0.010892
  dyy_13               = +0.000393
  dzz_13               = +0.001070
  dxx_23               = -0.015316
  dyy_23               = +0.008216
  dzz_23               = -0.000805
  dxy_12               = -0.008092
  dxz_12               = -0.036027
  dyz_12               = +0.046928
  dxy_21               = +0.003255
  dxz_21               = +0.023651
  dyz_21               = -0.041992
  dxy_13               = -0.012799
  dxz_13               = +0.021296
  dyz_13               = -0.029974
  dxy_31               = +0.011684
 

In [80]:
# ── Step 4: backprop through the CD solution ──────────────────────────────
# Example: gradient of ||x||^2 w.r.t. all parameters
loss_A = (x_t_Amat[n_single:]**2).sum()
loss_A.backward()
print(f"\nloss = ||x(2-body)||^2 = {loss_A.item():.6f}")
# print(f"d(loss)/d(Omega_1)      = {Omega_t[0].grad.item():+.6f}")
# print(f"d(loss)/d(Omega_2)      = {Omega_t[1].grad.item():+.6f}")
# print(f"d(loss)/d(dOmega_1/dlam)= {dOmega_t[0].grad.item():+.6f}")


loss = ||x(2-body)||^2 = 0.013114


In [81]:
optimizer = torch.optim.Adam([*Omega_t, *Mu_t, *Nu_t, *dOmega_t, *dMu_t, *dNu_t], lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, factor=0.5)

for step in range(1000):
    optimizer.zero_grad()
    M_t = make_M_torch(M_Om, M_Mu, M_Nu, M_U, Omega_t, Mu_t, Nu_t, U_t)
    b_t = make_b_torch(b_Om, b_Mu, b_Nu, dOmega_t, dMu_t, dNu_t)
    x_t = solve_cd_torch(M_t, b_t)
    loss = (x_t[n_single:] ** 2).sum()
    loss.backward()
    optimizer.step()
    scheduler.step(loss)

    if step % 100 == 0:
        print(f"step {step}: loss={loss.item():.6e}")


step 0: loss=1.311414e-02
step 100: loss=1.204539e-03
step 200: loss=4.297091e-04
step 300: loss=1.565875e-04
step 400: loss=5.399034e-05
step 500: loss=1.782156e-05
step 600: loss=6.081940e-06
step 700: loss=2.598271e-06
step 800: loss=1.613907e-06
step 900: loss=1.297805e-06


In [82]:
# fixed boundaries
Omega_0 = torch.zeros(n_atoms, dtype=torch.float64)
Omega_1 = torch.zeros(n_atoms, dtype=torch.float64)
Nu_0    = torch.full((n_atoms,), -10.0, dtype=torch.float64)
Nu_1    = torch.full((n_atoms,), +10.0, dtype=torch.float64)
Mu_0    = torch.zeros(n_atoms, dtype=torch.float64)
Mu_1    = torch.zeros(n_atoms, dtype=torch.float64)

# Free parameters (optimized) — midpoint amplitudes
Omega_mid = [torch.tensor(3.0, dtype=torch.float64, requires_grad=True) for _ in range(n_atoms)]
Nu_mid    = [torch.tensor(0.0, dtype=torch.float64, requires_grad=True) for _ in range(n_atoms)]
Mu_mid    = [torch.tensor(0.0, dtype=torch.float64, requires_grad=True) for _ in range(n_atoms)]

# Schedule at a given λ ∈ [0,1]
def get_params(lam):
    bump = lam * (1 - lam)  # vanishes at λ=0 and λ=1
    Omega_t = [(1-lam)*Omega_0[i] + lam*Omega_1[i] + bump*Omega_mid[i] for i in range(n_atoms)]
    Nu_t    = [(1-lam)*Nu_0[i]    + lam*Nu_1[i]    + bump*Nu_mid[i]    for i in range(n_atoms)]
    Mu_t    = [(1-lam)*Mu_0[i]    + lam*Mu_1[i]    + bump*Mu_mid[i]    for i in range(n_atoms)]
    return Omega_t, Nu_t, Mu_t


In [83]:
def get_dparams(lam):
    dbump = 1 - 2*lam
    dOmega_t = [Omega_1[i] - Omega_0[i] + dbump*Omega_mid[i] for i in range(n_atoms)]
    dNu_t    = [Nu_1[i]    - Nu_0[i]    + dbump*Nu_mid[i]    for i in range(n_atoms)]
    dMu_t    = [Mu_1[i]    - Mu_0[i]    + dbump*Mu_mid[i]    for i in range(n_atoms)]
    return dOmega_t, dNu_t, dMu_t


In [84]:
lam_values = torch.linspace(0.01, 0.99, 20)  # avoid endpoints (Ω=0 there)

optimizer = torch.optim.LBFGS([*Omega_mid, *Nu_mid, *Mu_mid], lr=0.1)

def closure():
    optimizer.zero_grad()
    total_loss = 0.0
    for lam in lam_values:
        Omega_t, Nu_t, Mu_t   = get_params(lam.item())
        dOmega_t, dNu_t, dMu_t = get_dparams(lam.item())
        M_t = make_M_torch(M_Om, M_Mu, M_Nu, M_U, Omega_t, Mu_t, Nu_t, U_t)
        b_t = make_b_torch(b_Om, b_Mu, b_Nu, dOmega_t, dMu_t, dNu_t)
        x_t = solve_cd_torch(M_t, b_t)
        total_loss += (x_t[n_single:] ** 2).sum()
    total_loss.backward()
    return total_loss


In [85]:
res = closure()
x_t

tensor([ 1.5665e-04, -2.2409e-04, -6.3012e-05,  1.9723e-04, -2.7733e-04,
         3.9591e-05,  1.8876e-04, -2.6347e-04, -3.1405e-05,  1.8255e-04,
        -1.9438e-04, -5.2758e-06,  1.6525e-04, -1.3721e-04,  9.1903e-07,
         1.9314e-04, -2.0852e-04,  2.7506e-06,  3.1366e-06, -2.4976e-04,
         3.9193e-04, -1.2392e-04, -1.8547e-04,  2.8560e-04, -2.9570e-05,
        -2.1327e-04,  3.5578e-04, -2.6493e-05, -2.0421e-04,  3.2724e-04,
        -1.2063e-04, -2.0461e-04,  2.9737e-04, -3.3372e-05, -2.0282e-04,
         2.9421e-04], dtype=torch.float64, grad_fn=<LinalgLstsqBackward0>)

In [86]:
# a_
# # ── Step 1: build tensors (reuse from cell 13 or rebuild here) ───────────
# n_cd = 4   # change to 3, 4, ...
# M_Om, M_Mu, M_Nu, M_U, b_Om, b_Mu, b_Nu = build_coefficient_tensors(n_cd)
# _, _, x_labels_cd = make_basis_ops(n_cd)

# Omega_t  = [torch.tensor(0.8 ** k,      dtype=torch.float64) for k in range(n_cd)]
# Mu_t     = [torch.tensor(0.0,           dtype=torch.float64) for _ in range(n_cd)]
# Nu_t     = [torch.tensor(0.5 * (-1)**k, dtype=torch.float64) for k in range(n_cd)]
# U_t      = {(i, j): torch.tensor(2.0,  dtype=torch.float64) for i, j in combinations(range(n_cd), 2)}

# dOmega_t = [torch.tensor(0.5 * (-1)**k, dtype=torch.float64) for k in range(n_cd)]
# dMu_t    = [torch.tensor(0.0,           dtype=torch.float64) for _ in range(n_cd)]
# dNu_t    = [torch.tensor(0.3 * (-1)**k, dtype=torch.float64) for k in range(n_cd)]

# M_t = make_M_torch(M_Om, M_Mu, M_Nu, M_U, Omega_t, Mu_t, Nu_t, U_t).detach()
# b_t = make_b_torch(b_Om, b_Mu, b_Nu, dOmega_t, dMu_t, dNu_t).detach()

# # ── Split columns: 1-body (a_k, b_k, c_k) vs 2-body ─────────────────────
# n_1body = 3 * n_cd
# M_1b = M_t[:, :n_1body]   # (N_eq, 3n)
# M_2b = M_t[:, n_1body:]   # (N_eq, n_2body)

# # ── Compute C and d via lstsq (avoids forming the full pseudo-inverse) ────
# # Solves  M_2b @ Z = [M_1b | b_t]  in the least-squares sense.
# # C = Z[:, :-1]  equivalent to  M_2b^+ @ M_1b
# # d = Z[:, -1]   equivalent to  M_2b^+ @ b_t
# rhs = torch.cat([M_1b, b_t.unsqueeze(1)], dim=1)   # (N_eq, 3n+1)
# Z   = torch.linalg.lstsq(M_2b, rhs).solution        # (n_2body, 3n+1)
# C   = Z[:, :-1]   # (n_2body, 3n)
# d   = Z[:, -1]    # (n_2body,)

# print(f"n_cd={n_cd}  |  M shape: {list(M_t.shape)}  |  n_1body={n_1body}  |  n_2body={M_2b.shape[1]}")
# print(f"||x_2body||² (x_1body=0): {d.pow(2).sum().item():.6e}")


In [87]:
# # ── Analytical minimiser of  ||d - C @ x_1body||²  ─────────────────────
# # x_1body_opt = pinv(C) @ d   (C is small: n_2body x 3n)
# x_1body_opt = torch.linalg.pinv(C) @ d
# x_2body_opt = d - C @ x_1body_opt

# print("Optimal 1-body CD coefficients:")
# for lbl, val in zip(x_labels_cd[:n_1body], x_1body_opt.tolist()):
#     print(f"  {lbl:10s} = {val:+.8f}")

# print(f"||x_2body||² before optimisation : {d.pow(2).sum().item():.6e}")
# print(f"||x_2body||² after  optimisation : {x_2body_opt.pow(2).sum().item():.6e}")

# print("2-body CD coefficients after optimisation:")
# for lbl, val in zip(x_labels_cd[n_1body:], x_2body_opt.tolist()):
#     print(f"  {lbl:15s} = {val:+.8f}")


In [88]:
# (gradient descent approach removed — analytical solution in cell above is exact and instant)

In [89]:
# (constrained optimisation sketch removed)